In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, t
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import KernelDensity, KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# =====================================================================
# IMPLEMENTAÇÃO CUSTOMIZADA DA JANELA DE PARZEN (KERNEL PRODUTO GAUSSIANO)
# =====================================================================
class ParzenBayesClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, bandwidth=1.0):
        self.bandwidth = bandwidth

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.kdes_ = {}
        self.priors_ = {}
        for c in self.classes_:
            X_c = X[y == c]
            self.priors_[c] = len(X_c) / len(X)
            # O Kernel Gaussiano padrão do Sklearn usa a métrica euclidiana,
            # mapeando perfeitamente o produto de kernels gaussianos univariados independentes.
            kde = KernelDensity(bandwidth=self.bandwidth, kernel='gaussian')
            kde.fit(X_c)
            self.kdes_[c] = kde
        return self

    def predict(self, X):
        log_probs = np.zeros((X.shape[0], len(self.classes_)))
        for idx, c in enumerate(self.classes_):
            # log P(x|w_i) + log P(w_i)
            log_probs[:, idx] = self.kdes_[c].score_samples(X) + np.log(self.priors_[c])
        return self.classes_[np.argmax(log_probs, axis=1)]

# =====================================================================
# CLASSICADOR DE VOTO MAJORITÁRIO (ENSEMBLE)
# =====================================================================
class MajorityVotingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators):
        self.estimators = estimators
    def fit(self, X, y):
        for est in self.estimators:
            est.fit(X, y)
        return self
    def predict(self, X):
        preds = np.array([est.predict(X) for est in self.estimators]) # Shape: (4, N)
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=preds)

# =====================================================================
# CARREGAMENTO E PRÉ-PROCESSAMENTO DO DATASET IONOSPHERE
# =====================================================================
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data"
data = pd.read_csv(url, header=None)
X = data.iloc[:, :-1].values
X = X[:, X.var(axis=0) > 0] # Remoção de colunas com variância zero
X_scaled = StandardScaler().fit_transform(X)

y_v1 = np.where(data.iloc[:, -1].values == 'g', 1, 0) # Classes originais a priori
y_v2 = np.random.choice([0, 1], size=len(y_v1))      # Substituir pelo vetor de clusters obtido no KCM-G-H

# =====================================================================
# PIPELINE DE VALIDAÇÃO CRUZADA E TUNING DE HIPERPARÂMETROS
# =====================================================================
def executar_experimento(X_data, y_data, nome_versao):
    print(f"\nIniciando Avaliação: {nome_versao}")

    nomes_modelos = ['GaussianNB', 'kNN', 'Parzen', 'LogReg', 'VotoMajoritario']
    historico = {m: {'erro': [], 'prec': [], 'rec': [], 'f1': []} for m in nomes_modelos}

    # Validação Cruzada Externa: 30 repetições de 10-folds
    rkf_outer = RepeatedStratifiedKFold(n_splits=10, n_repeats=30, random_state=42)

    for fold_idx, (train_idx, test_idx) in enumerate(rkf_outer.split(X_data, y_data)):
        X_train_out, X_test_out = X_data[train_idx], X_data[test_idx]
        y_train_out, y_test_out = y_data[train_idx], y_data[test_idx]

        modelos_sintonizados = {}

        # i) Classificador Bayesiano Gaussiano (Sem hiperparâmetros de sintonia)
        gnb = GaussianNB().fit(X_train_out, y_train_out)
        modelos_sintonizados['GaussianNB'] = gnb

        # Validação Cruzada Interna (5-folds) para ajuste de hiperparâmetros dos demais modelos
        kf_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        # ii) Tuning do k-NN (Ajustando k E as 3 Distâncias especificadas)
        best_knn_score = -1
        best_knn_params = {}
        for k in [3, 5, 7]:
            for dist in ['euclidean', 'manhattan', 'chebyshev']: # Respeitando City-Block e Chebyshev
                scores = []
                for tr_in, val_in in kf_inner.split(X_train_out, y_train_out):
                    knn_tmp = KNeighborsClassifier(n_neighbors=k, metric=dist)
                    knn_tmp.fit(X_train_out[tr_in], y_train_out[tr_in])
                    scores.append(accuracy_score(y_train_out[val_in], knn_tmp.predict(X_train_out[val_in])))
                if np.mean(scores) > best_knn_score:
                    best_knn_score = np.mean(scores)
                    best_knn_params = {'n_neighbors': k, 'metric': dist}
        modelos_sintonizados['kNN'] = KNeighborsClassifier(**best_knn_params).fit(X_train_out, y_train_out)

        # iii) Tuning da Janela de Parzen (Ajustando a janela h)
        best_parzen_score = -1
        best_h = 1.0
        for h in [0.1, 0.5, 1.0, 1.5, 2.0]:
            scores = []
            for tr_in, val_in in kf_inner.split(X_train_out, y_train_out):
                parzen_tmp = ParzenBayesClassifier(bandwidth=h)
                parzen_tmp.fit(X_train_out[tr_in], y_train_out[tr_in])
                scores.append(accuracy_score(y_train_out[val_in], parzen_tmp.predict(X_train_out[val_in])))
            if np.mean(scores) > best_parzen_score:
                best_parzen_score = np.mean(scores)
                best_h = h
        modelos_sintonizados['Parzen'] = ParzenBayesClassifier(bandwidth=best_h).fit(X_train_out, y_train_out)

        # iv) Tuning da Regressão Logística
        best_lr_score = -1
        best_c = 1.0
        for c in [0.01, 0.1, 1.0, 10.0]:
            scores = []
            for tr_in, val_in in kf_inner.split(X_train_out, y_train_out):
                lr_tmp = LogisticRegression(C=c, max_iter=1000)
                lr_tmp.fit(X_train_out[tr_in], y_train_out[tr_in])
                scores.append(accuracy_score(y_train_out[val_in], lr_tmp.predict(X_train_out[val_in])))
            if np.mean(scores) > best_lr_score:
                best_lr_score = np.mean(scores)
                best_c = c
        modelos_sintonizados['LogReg'] = LogisticRegression(C=best_c, max_iter=1000).fit(X_train_out, y_train_out)

        # v) Regra do Voto Majoritário baseado nos 4 anteriores
        lista_estimadores = [modelos_sintonizados[m] for m in ['GaussianNB', 'kNN', 'Parzen', 'LogReg']]
        modelos_sintonizados['VotoMajoritario'] = MajorityVotingClassifier(lista_estimadores).fit(X_train_out, y_train_out)

        # Coleta de Métricas no conjunto de Teste Externo
        for name, model in modelos_sintonizados.items():
            preds = model.predict(X_test_out)
            historico[name]['erro'].append(1.0 - accuracy_score(y_test_out, preds))
            historico[name]['prec'].append(precision_score(y_test_out, preds, average='macro', zero_division=0))
            historico[name]['rec'].append(recall_score(y_test_out, preds, average='macro', zero_division=0))
            historico[name]['f1'].append(f1_score(y_test_out, preds, average='macro', zero_division=0))

    return historico

# Execução do pipeline completo
historico_v1 = executar_experimento(X_scaled, y_v1, "Versão 1 - Original")


Iniciando Avaliação: Versão 1 - Original
